# Limpeza e padronização dos dados

Nesta etapa, as bases são tratadas e padronizadas para garantir consistência de estrutura, tipos e valores ausentes, preparando os dados para a integração e as análises posteriores.

## Tratamento da base de rendimento

A base de rendimento é consolidada para o período de 2018 a 2023, com padronização dos nomes das colunas, tratamento de valores ausentes, conversão de tipos e aplicação do recorte de escolas públicas de Porto Alegre.

In [5]:
from pathlib import Path
import pandas as pd

ANOS = tuple(range(2018, 2024))  # 2018 até 2023, como no notebook 01

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW = PROJECT_ROOT / "data" / "raw"
PROCESSED = PROJECT_ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

In [6]:
import pandas as pd

def arquivo_media_alunos(ano):
    pasta = RAW / "media_alunos_turma"

    arquivos = [
        arq for arq in pasta.rglob("*.xlsx")
        if str(ano) in str(arq)
    ]

    if not arquivos:
        raise FileNotFoundError(
            f"Arquivo de Média de Alunos não encontrado para {ano}"
        )

    return arquivos[0]


def encontrar_header_media(arquivo):
    xls = pd.ExcelFile(arquivo)
    aba = xls.sheet_names[0]

    amostra = pd.read_excel(
        arquivo,
        sheet_name=aba,
        header=None,
        nrows=20,
        dtype=str
    )

    for indice, linha in amostra.iterrows():
        valores = set(linha.dropna().astype(str))

        if "NO_ENTIDADE" in valores or "CO_ENTIDADE" in valores:
            return aba, indice

    raise ValueError(
        f"Cabeçalho técnico não encontrado em {arquivo}"
    )


def carregar_media_padronizada(ano):
    arquivo = arquivo_media_alunos(ano)
    aba, header = encontrar_header_media(arquivo)

    if ano == 2018:
        colunas = [
            "NO_ENTIDADE",
            "CO_ENTIDADE",
            "CO_MUNICIPIO",
            "ATU_FUN",
            "ATU_F14",
            "ATU_F04",
        ]

        mapa = {
            "NO_ENTIDADE": "codigo_escola",
            "CO_ENTIDADE": "nome_escola",
            "CO_MUNICIPIO": "municipio",
            "ATU_FUN": "media_alunos_fund",
            "ATU_F14": "media_alunos_ai",
            "ATU_F04": "media_alunos_af",
        }

    else:
        colunas = [
            "CO_ENTIDADE",
            "NO_ENTIDADE",
            "NO_MUNICIPIO",
            "FUN_CAT_0",
            "FUN_AI_CAT_0",
            "FUN_AF_CAT_0",
        ]

        mapa = {
            "CO_ENTIDADE": "codigo_escola",
            "NO_ENTIDADE": "nome_escola",
            "NO_MUNICIPIO": "municipio",
            "FUN_CAT_0": "media_alunos_fund",
            "FUN_AI_CAT_0": "media_alunos_ai",
            "FUN_AF_CAT_0": "media_alunos_af",
        }

    df = pd.read_excel(
        arquivo,
        sheet_name=aba,
        header=header,
        usecols=colunas,
        dtype=str
    )

    df = df.rename(columns=mapa)
    df["ano"] = ano

    return df


# 1. Carregamento e concatenação
media_alunos = pd.concat(
    [carregar_media_padronizada(ano) for ano in ANOS],
    ignore_index=True
)


# 2. Tratamento de ausências
colunas_media = [
    "media_alunos_fund",
    "media_alunos_ai",
    "media_alunos_af",
]

media_alunos[colunas_media] = (
    media_alunos[colunas_media]
    .replace("--", pd.NA)
)


# 3. Conversão das médias para numérico
for coluna in colunas_media:
    media_alunos[coluna] = pd.to_numeric(
        media_alunos[coluna],
        errors="coerce"
    )


# 4. Padronização de tipos
media_alunos["ano"] = (
    pd.to_numeric(
        media_alunos["ano"],
        errors="coerce"
    )
    .astype("Int64")
)

media_alunos["codigo_escola"] = (
    media_alunos["codigo_escola"]
    .astype("string")
    .str.strip()
)

for coluna in [
    "nome_escola",
    "municipio",
]:
    media_alunos[coluna] = (
        media_alunos[coluna]
        .astype("string")
        .str.strip()
    )


# 5. Recorte de Porto Alegre
media_alunos = media_alunos[
    media_alunos["municipio"] == "Porto Alegre"
].copy()


# 6. Validação final
print(media_alunos.shape)

display(media_alunos.head())

display(
    media_alunos.groupby("ano").agg(
        registros=("codigo_escola", "size"),
        media_disponivel=("media_alunos_fund", "count"),
        media_ausente=(
            "media_alunos_fund",
            lambda s: s.isna().sum()
        )
    )
)

(5651, 7)


,municipio,codigo_escola,nome_escola,media_alunos_fund,media_alunos_ai,media_alunos_af,ano
163437,Porto Alegre,43000452,ESC ENS MEDIO CESI ZONA SUL,32.9,31.2,36.3,2018
163438,Porto Alegre,43000479,E E IND ENS FUN TUPE PAN,6.0,NaN,NaN,2018
163439,Porto Alegre,43000487,ESC EDUC PROFISSIONAL CECILIA MEIRELES,NaN,NaN,NaN,2018
163440,Porto Alegre,43001190,IFRS - CAMPUS PORTO ALEGRE RESTINGA,NaN,NaN,NaN,2018
163441,Porto Alegre,43001254,E E IND ENS FUN PINDO POTY,1.5,1.0,NaN,2018


,registros,media_disponivel,media_ausente
ano,,,
2018,987,359,628
2019,973,355,618
2020,950,354,596
2021,923,352,571
2022,922,353,569
2023,896,351,545


## Tratamento da base de média de alunos por turma

A base é consolidada para o período de 2018 a 2023, com padronização dos identificadores e indicadores, tratamento de valores ausentes, conversão de tipos e recorte das escolas de Porto Alegre.

In [7]:

def arquivo_media_alunos(ano):
    pasta = RAW / "media_alunos_turma"

    arquivos = [
        arq for arq in pasta.rglob("*.xlsx")
        if str(ano) in str(arq)
    ]

    if not arquivos:
        raise FileNotFoundError(f"Arquivo de Média de Alunos não encontrado para {ano}")

    return arquivos[0]


def encontrar_header_media(arquivo):
    xls = pd.ExcelFile(arquivo)
    aba = xls.sheet_names[0]

    amostra = pd.read_excel(
        arquivo,
        sheet_name=aba,
        header=None,
        nrows=20,
        dtype=str
    )

    for indice, linha in amostra.iterrows():
        valores = set(linha.dropna().astype(str))

        if "NO_ENTIDADE" in valores or "CO_ENTIDADE" in valores:
            return aba, indice

    raise ValueError(f"Cabeçalho técnico não encontrado em {arquivo}")


def carregar_media_padronizada(ano):
    arquivo = arquivo_media_alunos(ano)
    aba, header = encontrar_header_media(arquivo)

    df = pd.read_excel(
        arquivo,
        sheet_name=aba,
        header=header,
        dtype=str
    )

    # 2018 possui inversão nos nomes técnicos dos identificadores
    if ano == 2018:
        mapa = {
            "NO_ENTIDADE": "codigo_escola",
            "CO_ENTIDADE": "nome_escola",
            "CO_MUNICIPIO": "municipio",
            "ATU_FUN": "media_alunos_fund",
            "ATU_F14": "media_alunos_ai",
            "ATU_F04": "media_alunos_af",
        }

    else:
        mapa = {
            "CO_ENTIDADE": "codigo_escola",
            "NO_ENTIDADE": "nome_escola",
            "NO_MUNICIPIO": "municipio",
            "FUN_CAT_0": "media_alunos_fund",
            "FUN_AI_CAT_0": "media_alunos_ai",
            "FUN_AF_CAT_0": "media_alunos_af",
        }

    df = df[list(mapa)].rename(columns=mapa)
    df["ano"] = ano

    return df


# 1. Carregamento e concatenação
media_alunos = pd.concat(
    [carregar_media_padronizada(ano) for ano in ANOS],
    ignore_index=True
)

# 2. Tratamento de ausências
colunas_media = [
    "media_alunos_fund",
    "media_alunos_ai",
    "media_alunos_af",
]

media_alunos[colunas_media] = (
    media_alunos[colunas_media]
    .replace("--", pd.NA)
)

# 3. Conversão de tipos
for coluna in colunas_media:
    media_alunos[coluna] = pd.to_numeric(
        media_alunos[coluna],
        errors="coerce"
    )

media_alunos["ano"] = (
    pd.to_numeric(media_alunos["ano"], errors="coerce")
    .astype("Int64")
)

media_alunos["codigo_escola"] = (
    media_alunos["codigo_escola"]
    .astype("string")
    .str.strip()
)

for coluna in ["nome_escola", "municipio"]:
    media_alunos[coluna] = (
        media_alunos[coluna]
        .astype("string")
        .str.strip()
    )

# 4. Recorte de Porto Alegre
media_alunos = media_alunos[
    media_alunos["municipio"] == "Porto Alegre"
].copy()

# 5. Validação final
print(media_alunos.shape)

display(media_alunos.head())

display(
    media_alunos.groupby("ano").agg(
        registros=("codigo_escola", "size"),
        media_disponivel=("media_alunos_fund", "count"),
        media_ausente=("media_alunos_fund", lambda s: s.isna().sum())
    )
)

(5651, 7)


,codigo_escola,nome_escola,municipio,media_alunos_fund,media_alunos_ai,media_alunos_af,ano
163437,43000452,ESC ENS MEDIO CESI ZONA SUL,Porto Alegre,32.9,31.2,36.3,2018
163438,43000479,E E IND ENS FUN TUPE PAN,Porto Alegre,6.0,NaN,NaN,2018
163439,43000487,ESC EDUC PROFISSIONAL CECILIA MEIRELES,Porto Alegre,NaN,NaN,NaN,2018
163440,43001190,IFRS - CAMPUS PORTO ALEGRE RESTINGA,Porto Alegre,NaN,NaN,NaN,2018
163441,43001254,E E IND ENS FUN PINDO POTY,Porto Alegre,1.5,1.0,NaN,2018


,registros,media_disponivel,media_ausente
ano,,,
2018,987,359,628
2019,973,355,618
2020,950,354,596
2021,923,352,571
2022,922,353,569
2023,896,351,545


In [11]:
# Carregamento e tratamento — Rendimento escolar

def carregar_rendimento_padronizado(ano):
    arquivos = sorted(
        arq for arq in (RAW / "rendimento").rglob("*.xlsx")
        if str(ano) in str(arq) and not arq.name.startswith("~$")
    )

    if not arquivos:
        raise FileNotFoundError(
            f"Arquivo de rendimento não encontrado para {ano}"
        )

    if len(arquivos) > 1:
        raise ValueError(
            f"Mais de um arquivo de rendimento para {ano}: {arquivos}"
        )

    # Os nomes das taxas mudam a partir de 2021.
    mapa = {
        "CO_ENTIDADE": "codigo_escola",
        "NO_ENTIDADE": "nome_escola",
        "NO_MUNICIPIO": "municipio",
        "SG_UF": "uf",
        "tap_FUN" if ano <= 2020 else "1_CAT_FUN": "taxa_aprovacao",
        "tre_FUN" if ano <= 2020 else "2_CAT_FUN": "taxa_reprovacao",
        "tab_FUN" if ano <= 2020 else "3_CAT_FUN": "taxa_abandono",
    }

    df = pd.read_excel(
        arquivos[0],
        sheet_name="ESCOLAS",
        header=8,
        usecols=list(mapa),
        dtype=str,
    ).rename(columns=mapa)

    df["ano"] = ano

    # Padronização dos textos e do código da escola.
    for coluna in ["codigo_escola", "nome_escola", "municipio", "uf"]:
        df[coluna] = df[coluna].astype("string").str.strip()

    df["codigo_escola"] = df["codigo_escola"].str.replace(
        r"\.0$", "", regex=True
    )

    # Recorte de Porto Alegre — RS.
    df = df[
        df["municipio"].eq("Porto Alegre")
        & df["uf"].eq("RS")
    ].copy()

    # Conversão das taxas, preservando ausências como valores nulos.
    for coluna in ["taxa_aprovacao", "taxa_reprovacao", "taxa_abandono"]:
        valores = (
            df[coluna]
            .astype("string")
            .str.strip()
            .replace({"--": pd.NA, "": pd.NA})
            .str.replace(",", ".", regex=False)
        )
        df[coluna] = pd.to_numeric(valores, errors="coerce")

    return df


rendimento = pd.concat(
    [carregar_rendimento_padronizado(ano) for ano in ANOS],
    ignore_index=True,
)

rendimento["ano"] = rendimento["ano"].astype("Int64")

print("Rendimento:", rendimento.shape)
display(rendimento.head())

display(
    rendimento.groupby("ano").agg(
        registros=("codigo_escola", "size"),
        escolas=("codigo_escola", "nunique"),
        aprovacao_disponivel=("taxa_aprovacao", "count"),
    )
)

Rendimento: (2292, 8)


,uf,municipio,codigo_escola,nome_escola,taxa_aprovacao,taxa_reprovacao,taxa_abandono,ano
0,RS,Porto Alegre,43000452,ESC ENS MEDIO CESI ZONA SUL,98.4,1.6,0.0,2018
1,RS,Porto Alegre,43000479,E E IND ENS FUN TUPE PAN,91.7,8.3,0.0,2018
2,RS,Porto Alegre,43001190,IFRS - CAMPUS PORTO ALEGRE RESTINGA,<NA>,<NA>,<NA>,2018
3,RS,Porto Alegre,43001254,E E IND ENS FUN PINDO POTY,100.0,0.0,0.0,2018
4,RS,Porto Alegre,43002803,COLEGIO MARISTA IRMAO JAIME BIAZUS,<NA>,<NA>,<NA>,2018


,registros,escolas,aprovacao_disponivel
ano,,,
2018,385,385,357
2019,382,382,353
2020,381,381,348
2021,380,380,338
2022,381,381,341
2023,383,383,346


## Tratamento da base do Censo Escolar

A base do Censo Escolar é consolidada para o período de 2018 a 2023, com aplicação do recorte de Porto Alegre, rede pública e escolas com oferta de Ensino Fundamental. Também são padronizados identificadores, tipos e variáveis de infraestrutura, matrícula, docentes e turmas.

In [12]:


def arquivo_censo(ano):
    pasta = (
        RAW
        / "censo_escolar"
        / f"microdados_ed_basica_{ano}"
        / "dados"
    )

    arquivos = [
        arq for arq in pasta.iterdir()
        if arq.is_file()
        and arq.suffix.lower() == ".csv"
        and "microdados_ed_basica" in arq.name.lower()
    ]

    if not arquivos:
        raise FileNotFoundError(
            f"Arquivo do Censo não encontrado para {ano}"
        )

    return arquivos[0]

COLUNAS_CENSO = [
    "NU_ANO_CENSO",
    "CO_ENTIDADE",
    "NO_ENTIDADE",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "SG_UF",
    "TP_DEPENDENCIA",
    "TP_LOCALIZACAO",
    "TP_SITUACAO_FUNCIONAMENTO",
    "IN_FUND",

    "IN_AGUA_POTAVEL",
    "IN_AGUA_REDE_PUBLICA",
    "IN_ENERGIA_REDE_PUBLICA",
    "IN_ESGOTO_REDE_PUBLICA",
    "IN_BANHEIRO",

    "IN_BIBLIOTECA",
    "IN_BIBLIOTECA_SALA_LEITURA",
    "IN_SALA_LEITURA",
    "IN_LABORATORIO_CIENCIAS",
    "IN_LABORATORIO_INFORMATICA",
    "IN_COMPUTADOR",
    "IN_DESKTOP_ALUNO",
    "IN_COMP_PORTATIL_ALUNO",
    "IN_TABLET_ALUNO",
    "IN_INTERNET",
    "IN_INTERNET_ALUNOS",
    "IN_INTERNET_APRENDIZAGEM",
    "IN_BANDA_LARGA",

    "IN_BANHEIRO_PNE",
    "IN_ACESSIBILIDADE_RAMPAS",
    "IN_ACESSIBILIDADE_CORRIMAO",
    "IN_ACESSIBILIDADE_PISOS_TATEIS",
    "IN_ACESSIBILIDADE_SINAL_SONORO",
    "IN_ACESSIBILIDADE_SINAL_TATIL",
    "IN_ACESSIBILIDADE_SINAL_VISUAL",
    "IN_ACESSIBILIDADE_INEXISTENTE",

    "QT_MAT_FUND",
    "QT_MAT_FUND_AI",
    "QT_MAT_FUND_AF",
    "QT_DOC_FUND",
    "QT_DOC_FUND_AI",
    "QT_DOC_FUND_AF",
    "QT_TUR_FUND",
    "QT_TUR_FUND_AI",
    "QT_TUR_FUND_AF",
]


def carregar_censo_tratado(ano):
    arquivo = arquivo_censo(ano)
    partes = []

    for chunk in pd.read_csv(
        arquivo,
        sep=";",
        encoding="latin1",
        dtype=str,
        usecols=COLUNAS_CENSO,
        chunksize=50_000,
        low_memory=False
    ):
        # Recorte aplicado já durante a leitura
        chunk = chunk[
            (chunk["NO_MUNICIPIO"] == "Porto Alegre") &
            (chunk["SG_UF"] == "RS") &
            (chunk["TP_DEPENDENCIA"].isin(["1", "2", "3"])) &
            (chunk["IN_FUND"] == "1")
        ].copy()

        partes.append(chunk)

    return pd.concat(partes, ignore_index=True)


# 1. Carregamento e concatenação
censo = pd.concat(
    [carregar_censo_tratado(ano) for ano in ANOS],
    ignore_index=True
)

# 2. Padronização dos nomes principais
censo = censo.rename(columns={
    "NU_ANO_CENSO": "ano",
    "CO_ENTIDADE": "codigo_escola",
    "NO_ENTIDADE": "nome_escola",
    "CO_MUNICIPIO": "codigo_municipio",
    "NO_MUNICIPIO": "municipio",
    "SG_UF": "uf",
    "TP_DEPENDENCIA": "dependencia_administrativa",
    "TP_LOCALIZACAO": "localizacao",
    "TP_SITUACAO_FUNCIONAMENTO": "situacao_funcionamento",
})

# 3. Conversão dos identificadores e textos
censo["ano"] = pd.to_numeric(
    censo["ano"],
    errors="coerce"
).astype("Int64")

censo["codigo_escola"] = (
    censo["codigo_escola"]
    .astype("string")
    .str.strip()
)

censo["codigo_municipio"] = (
    censo["codigo_municipio"]
    .astype("string")
    .str.strip()
)

for coluna in ["nome_escola", "municipio", "uf"]:
    censo[coluna] = (
        censo[coluna]
        .astype("string")
        .str.strip()
    )

# 4. Padronização da dependência administrativa
mapa_dependencia = {
    "1": "Federal",
    "2": "Estadual",
    "3": "Municipal",
}

censo["dependencia_administrativa"] = (
    censo["dependencia_administrativa"]
    .map(mapa_dependencia)
    .astype("string")
)

# 5. Conversão das variáveis numéricas
colunas_numericas = [
    coluna for coluna in censo.columns
    if coluna.startswith(("IN_", "QT_"))
]

for coluna in colunas_numericas:
    censo[coluna] = pd.to_numeric(
        censo[coluna],
        errors="coerce"
    )

# 6. Validação final
print(censo.shape)

display(censo.head())

display(
    censo.groupby("ano").agg(
        registros=("codigo_escola", "size"),
        escolas=("codigo_escola", "nunique")
    )
)

(1621, 45)


,ano,uf,municipio,codigo_municipio,codigo_escola,nome_escola,dependencia_administrativa,localizacao,situacao_funcionamento,IN_AGUA_POTAVEL,...,IN_FUND,QT_MAT_FUND,QT_MAT_FUND_AI,QT_MAT_FUND_AF,QT_DOC_FUND,QT_DOC_FUND_AI,QT_DOC_FUND_AF,QT_TUR_FUND,QT_TUR_FUND_AI,QT_TUR_FUND_AF
0,2018,RS,Porto Alegre,4314902,43000479,E E IND ENS FUN TUPE PAN,Estadual,1,1,0,...,1,12,12,0,2,0,2,2,0,2
1,2018,RS,Porto Alegre,4314902,43001254,E E IND ENS FUN PINDO POTY,Estadual,1,1,0,...,1,3,3,0,2,1,1,2,1,1
2,2018,RS,Porto Alegre,4314902,43005047,E E IND ENS FUN KA AGUY MIRI,Estadual,1,1,0,...,1,6,6,0,2,0,2,2,0,2
3,2018,RS,Porto Alegre,4314902,43048200,EMEF PORTO NOVO,Municipal,1,1,0,...,1,350,234,116,23,15,8,13,9,4
4,2018,RS,Porto Alegre,4314902,43104932,COLEGIO DE APLICACAO UFRGS,Federal,1,1,0,...,1,296,100,196,50,11,39,12,5,7


,registros,escolas
ano,,
2018,276,276
2019,272,272
2020,271,271
2021,268,268
2022,267,267
2023,267,267


## Validação das bases tratadas

Após o tratamento, são verificadas as dimensões das bases e a presença de valores ausentes nas chaves `ano` e `codigo_escola`, garantindo que os dados estejam consistentes para a etapa de integração.

In [13]:
print("Rendimento:", rendimento.shape)
print("Média de alunos:", media_alunos.shape)
print("Censo:", censo.shape)

print("\nNulos nas chaves principais:")

print(
    "Rendimento:",
    rendimento[["ano", "codigo_escola"]].isna().sum().to_dict()
)

print(
    "Média de alunos:",
    media_alunos[["ano", "codigo_escola"]].isna().sum().to_dict()
)

print(
    "Censo:",
    censo[["ano", "codigo_escola"]].isna().sum().to_dict()
)

Rendimento: (2292, 8)
Média de alunos: (5651, 7)
Censo: (1621, 45)

Nulos nas chaves principais:
Rendimento: {'ano': 0, 'codigo_escola': 0}
Média de alunos: {'ano': 0, 'codigo_escola': 0}
Censo: {'ano': 0, 'codigo_escola': 0}


## Exportação das bases tratadas

As bases padronizadas são salvas em `data/processed` para utilização nas etapas seguintes, preservando os arquivos originais em `data/raw`.

In [14]:
rendimento.to_csv(
    PROCESSED / "rendimento_tratado.csv",
    index=False
)

media_alunos.to_csv(
    PROCESSED / "media_alunos_tratado.csv",
    index=False
)

censo.to_csv(
    PROCESSED / "censo_tratado.csv",
    index=False
)

print("Bases tratadas salvas em data/processed/")

Bases tratadas salvas em data/processed/
